[Guide Index](README.md)

# 05.02 The StarLayer SHACL engine

`starlayer.shacl` is StarLayer's SHACL engine. 

The engine can be used in two ways: through the methods of the `StarShaclProcessor` and through various functions. Most actions of the engine can use both approaches.  The various SHACL notebooks primarily demonstrate the use of the processor.  

This notebook describes the various functions and methods of the engine and where each is discussed in the guide.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph
from starlayer.graph.config import set_default_prefixes
from starlayer.shacl import StarShaclProcessor, target_nodes, validate

# EX and SH are starlayer.graph Namespaces: their terms are pyoxigraph NamedNodes, used to look terms up in the results below
# (StarLayerGraphs, see section 05.01.2); the StarLayerGraph inputs are built by parsing Turtle.
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

# Every StarLayerGraph made after this has these prefixes, so results, queries and the report text use short names such as
# ex:alice even for a graph built in Python rather than parsed from Turtle (a prefix the Turtle declares itself still applies).
set_default_prefixes({
    "ex": "http://example.org/",
    "sh": "http://www.w3.org/ns/shacl#",
    "shnex": "http://www.w3.org/ns/shacl-node-expr#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
})


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses."""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 05.02.1 Index of functions and methods

For the main operations of the engine, the table shows the function, whether it is also a method of a `StarShaclProcessor` ✅, what it does, and where it is discussed. A method uses the graphs and settings the processor holds. 

Everything in the table, and `StarShaclProcessor`, are imported from `starlayer.shacl`.

| Function | Method | What it does | Discussed in |
|---|:-:|---|---|
| **Validating, evaluating and running rules** | | | |
| `validate()` | ✅ | Checks a data graph against a shapes graph and returns a `ValidationReport`. | [05.01](05.01-validating-with-shacl-1.2.ipynb) |
| `validate_each()` | ✅ | Validates each graph of a dataset against the shapes graph. | [05.01.4.1](05.01-validating-with-shacl-1.2.ipynb) |
| `evaluate()` | ✅ | The values of a node or nodes for the properties of a shape, with `sh:values` calculated and `sh:defaultValue` used, as a graph. | [05.11](05.11-shacl-evaluate.ipynb) |
| `apply_rules()` | ✅ | Runs the shapes' rules over the data until nothing new is inferred. Returns the inferred triples. The data graph is not changed. | [05.06](05.06-shacl-inference-rules.ipynb) |
| `target_nodes()` | ✅ | The nodes a shape targets in the data graph, which are the nodes `validate()` checks for that shape. | [05.01.3.6](05.01-validating-with-shacl-1.2.ipynb), [05.03](05.03-shapes-as-classes.ipynb) |
| `target_shapes()` | ✅ | The shapes that target a node, the reverse of `target_nodes()`. | [05.01.3.6](05.01-validating-with-shacl-1.2.ipynb), [05.03](05.03-shapes-as-classes.ipynb) |
| **Writing and extracting shapes** | | | |
| `derive_shape()` |  | Builds one node shape from a class or a node in the data graph, or brings an existing shape up to date with the data. Used to start designing a shape from data. | section 05.02.3 |
| `extract_subgraph()` | ✅ | The part of the data graph that one shape covers for a focus node, or that all the shapes of a shapes graph cover. | [05.09](05.09-shacl-subgraph-extraction.ipynb) |
| `close_shape()` | ✅ | A copy of the shapes graph in which a shape, and the shapes it reaches, are closed. | [05.09](05.09-shacl-subgraph-extraction.ipynb) |
| `extract_shapes_graph()` | ✅ | One named `sh:ShapesGraph` as a graph of its own, with its members and prefixes. | [05.08](05.08-shacl-profiling.ipynb) |
| **SHACL UI and editors** | | | |
| `select_widgets()` | ✅ | The widget for each property shape of a node shape, for a given role (editor or viewer). | [05.07](05.07-shacl-ui.ipynb) |
| `select_widget()` | ✅ | The widget for one property shape and given focus node, by the SHACL UI selection procedure. | [05.07](05.07-shacl-ui.ipynb) |
| `explain_widgets()` | ✅ | Every candidate widget for a property shape, and the steps that chose the selected widget. | [05.07](05.07-shacl-ui.ipynb) |
| `candidate_values()` | ✅ | What a form can offer for a field: the values the shape names, with the instances of a class and its subclasses. | [05.10](05.10-building-on-shacl.ipynb) |
| `value_form()` | ✅ | Whether a field holds one value, an RDF list, or either, read from its shape. | [05.10](05.10-building-on-shacl.ipynb) |
| `check_value()` | ✅ | Whether a value would be accepted as valid for a field of a node, before it is saved. | [05.10](05.10-building-on-shacl.ipynb) |
| `form_model()` | ✅ | The fields of a node's form, with their widgets, their values and source of each value. | [05.10](05.10-building-on-shacl.ipynb) |
| **Checking shapes** | | | |
| `meta_validate()` |  | Checks a shapes graph against SHACL's own shapes, or a proposed edit to a shapes graph, before it is made. | section 05.02.4 |
| `meta_scope()` |  | What a check of an edit would cover, and why. | section 05.02.4.2 |
| **Reading a report** | | | |
| `report_results()` |  | The results of a validation report. | [05.10](05.10-building-on-shacl.ipynb) |
| `diff_reports()` |  | Compares two validation reports and outputs the changes. | [05.10](05.10-building-on-shacl.ipynb) |
| `derive_conforms_to()` |  | Derives `sh:conformsToShapesGraph` from a validation report and an optional `specification_graph` | [05.08](05.08-shacl-profiling.ipynb) |


A function takes the graphs and optional parameters on every call. `StarShaclProcessor` binds a shapes graph, and optionally other graphs and settings, once, so that it can be re-used with different data graphs, and offers nearly all of the same operations as methods. 

The rest of this guide covers the functionality of the SHACL engine not covered elsewhere:

- how the processor is configured, and how a call overrides the configuration (section 05.02.2),
- how to derive a shape from data (section 05.02.3),
- how to check a shapes graph (section 05.02.4),
- the errors (section 05.02.5), and
- descriptive shape properties (section 05.02.6).

The functions and the processor take a `StarLayerGraph`, a `StarLayerDataset` or a `pyoxigraph.Store`. Anything else (e.g. an rdflib graph) raises a `TypeError` that names the argument.

The results and errors are classes: `ValidationReport` (the result of `validate()`), `RulesResult` (the result of `apply_rules()`), and the errors `ShapeError` (with `ShapeCompileError` and `MetaShaclError`) and `ExecutionError` (section 05.02.5). `SH` is a namespace of the SHACL terms.

## 05.02.2 Configuring the processor

A `StarShaclProcessor` is configured when it is created. Only the shapes graph is required: all other graphs and all settings are optional. Configurations set at creation and can be modified. Any configuration can be overridden during an individual method call without changing the configuration. From lowest to highest, the precedence is the built-in default, the processor's configuration, then the call. The functions have no configuration of their own: they take the same settings as keywords on each call.

The processor holds its graphs and does not copy them, so a change made to the data graph or the shapes graph is seen by the next method call. 

The graphs can be given when the processor is created.  Only the shapes graph is required.

| Graph | What it is |
|---|---|
| `shacl_graph` | the shapes graph |
| `ont_graph` | an ontology graph, read when an entailment regime is used ([05.01.4.3](05.01-validating-with-shacl-1.2.ipynb)) |
| `data_graph` | a default data graph, so that `validate()` or `evaluate()` can be called without one |
| `scoring_graph` | the widget scoring catalog for `select_widgets()` (see the [SHACL UI guide](05.07-shacl-ui.ipynb)) |

The settings are keywords after the graphs. Each is held by the processor and used by the calls that need it. Related settings are together: checking the shapes graph and its imports, entailment and the class hierarchy, conformance, and limits:

| Setting | Built-in default | What it does | Discussed in |
|---|---|---|---|
| `meta_shacl` | `False` | check the shapes graph against SHACL's own SHACL for shapes when the processor is created, and raise `MetaShaclError` if it does not conform | section 05.02.4 |
| `meta_shapes_extra` | none | additional shapes to be included in the meta_shacl check | section 05.02.4.1 |
| `ignore_imports`, `imports_loader` | `False`, none | how `owl:imports` of the shapes graph are resolved. An import of an ontology StarLayer has built in is skipped. Any other declared import that is not in the graph raises an error unless `imports_loader` is a function that supplies it or `ignore_imports=True` | section 05.02.2.2 |
| `inference` | none | an entailment regime IRI such as `ENTAILMENT.RDFS`, or several, applied to the data graph of every call | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb) |
| `recompute` | `"onChange"` | when the entailed triples of `inference` are computed again: `"onChange"` (when the graph has changed), `"never"` or `"always"`, as for a SPARQL query | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb) |
| `engine`, `timeout` | `"hermit"`, none | for the `OWL-Direct` regime only: the reasoner and its time limit, as for a SPARQL query | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb) |
| `subclass_of_in_shapes_graph` | `False` | also read `rdfs:subClassOf` from the shapes graph | [05.03.2](05.03-shapes-as-classes.ipynb) |
| `allow_warnings`, `allow_infos` | `False` | a warning (or an info) no longer makes `conforms` false | [05.01.3.7](05.01-validating-with-shacl-1.2.ipynb) |
| `max_iterations` | 1000 | limit on the rounds of `apply_rules()`, the method and the function | [05.06](05.06-shacl-inference-rules.ipynb), section 05.06.12 |
| `max_triples` | 5,000,000 | limit on the triples one run of `apply_rules()` may infer; `None` removes it | [05.06](05.06-shacl-inference-rules.ipynb), section 05.06.12 |

### 05.02.2.1 Overriding the configuration

The configuration of a processor is the starting point of each call. A setting has one of three values, and the highest of them is the one a call uses:

1. the built-in default of the setting (see the table above),
2. the processor's configuration: the value given when the processor was created, or set later by assigning the attribute (`myProcessor.allow_warnings = True`), which changes the configuration for every call after that, and
3. the call: a keyword passed to a method of the processor, such as `validate(allow_warnings=True)`, which applies to that call only and leaves the processor's configuration unchanged.

The same keywords are used at each level. A function has no configuration, so it uses built-in defaults and the keywords of the call if provided. A method call can also provide a value the processor lacks, such as a data graph when none is bound. This makes a processor a reusable base: it is configured once for what is usual, and a call override for exceptions.

Below, `ex:bob` has no name, which is a warning. With the built-in default, `allow_warnings=False`, a warning makes him not conform. Assigning `myProcessor.allow_warnings = True` changes the configuration for every call after that (conforms). A method call with `allow_warnings=False` overrides the processor's setting for that call only (does not conform), and the setting remains unchanged. The warning is in the validation results every time: `allow_warnings` changes only whether it makes `conforms` false.

In [2]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ; sh:severity sh:Warning ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob a ex:Person .
""", format="turtle")


def show_results(label, report):
    results = [f"{r.severity_name}: {r.messages[0].value}" for r in report.results]
    print(f"{label}: conforms={report.conforms}, results={results}")


# 1. the built-in default: allow_warnings is False, so a warning makes conforms false
myProcessor = StarShaclProcessor(shacl_graph=shapes, data_graph=data)
show_results("built-in default", myProcessor.validate())

# 2. the processor's configuration: assigning the attribute changes it for every call after that
myProcessor.allow_warnings = True
show_results("the processor's setting", myProcessor.validate())

# 3. a call overrides the setting for that call only
show_results("the call's setting", myProcessor.validate(allow_warnings=False))
print("the processor's setting after the call:", myProcessor.allow_warnings)

built-in default: conforms=False, results=['Warning: Needs at least 1 value for ex:name']
the processor's setting: conforms=True, results=['Warning: Needs at least 1 value for ex:name']
the call's setting: conforms=False, results=['Warning: Needs at least 1 value for ex:name']
the processor's setting after the call: True


### 05.02.2.2 Imports of the shapes graph: `ignore_imports` and `imports_loader`

One shapes graph can import another shapes graph with `owl:imports`. The SHACL spec says a processor should follow the imports before it validates. An import that is not in the shapes graph raises a `ExecutionError` of kind `import` at the first `validate()`. Skipping an import would drop the shapes it supplies, so any validation of a data graph would be incomplete.  

There are three ways to handle an import that is not already in the shapes graph:

- **Manually merge the imported graph into the shapes graph.** An import counts as present when its IRI is the subject of triples in the shapes graph.
- **Give a `imports_loader`.** It is a function that is called with the IRI of the import, a `pyoxigraph.NamedNode`, and returns a graph, or `None` when it has none. The imported shapes then take part as if they were in the shapes graph.
- **Set `ignore_imports=True`.** The imports are not followed, and the shapes they would have supplied are not used. It is used for imports that need no loading: the modules are already merged into the shapes graph but the `owl:imports` statements were kept, or the import is only a declaration.

An `owl:imports` of an ontology StarLayer already has built in (the SHACL vocabulary, SHACL UI, SHACL-SHACL and SKOS, written with or without the final `#` or `/`) needs no loader and no network: it is skipped, never passed to the `imports_loader`, and adds no triples to the shapes graph. Any other import that is not in the shapes graph still raises a `ExecutionError` of kind `import` unless a loader is passed or `ignore_imports=True`. 

The [SHACL profiling guide](05.08-shacl-profiling.ipynb) covers the packaging conventions for `owl:imports` and `sh:ShapesGraph`.

Below, `ex:PersonShape` requires every person to conform to `ex:CommonShape`, which is in a graph that the shapes graph imports. Without a loader, validating raises an error. With a loader that returns the common shapes, `ex:alice`, who has no name, does not conform. With `ignore_imports=True` the common shapes are not used, so she conforms.

In [3]:
from starlayer.errors import ExecutionError

main = StarLayerGraph()
main.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Main a owl:Ontology ; owl:imports <http://example.org/common> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:node ex:CommonShape .
""", format="turtle")

common = StarLayerGraph()
common.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:CommonShape a sh:NodeShape ; sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

people = StarLayerGraph()
people.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")

# the import is not in the shapes graph, and nothing supplies it
try:
    StarShaclProcessor(shacl_graph=main).validate(data_graph=people)
except ExecutionError as error:
    print("no loader:", error)

# a loader supplies the imported graph for its IRI
def loader(iri):
    return common if iri.value == "http://example.org/common" else None

print("with a loader, conforms:", StarShaclProcessor(shacl_graph=main, imports_loader=loader).validate(data_graph=people).conforms)

# the imports are not followed
print("with ignore_imports, conforms:", StarShaclProcessor(shacl_graph=main, ignore_imports=True).validate(data_graph=people).conforms)

no loader: shacl_graph declares owl:imports ['http://example.org/common'] that aren't already locally satisfied, but no shapes_graph_loader was given to resolve them. Pass shapes_graph_loader=... to resolve them, or ignore_imports=True if you've already resolved them yourself (or don't need them).
with a loader, conforms: False
with ignore_imports, conforms: True


## 05.02.3 Deriving shapes from data

`derive_shape()` is a function of `starlayer.shacl`, with no method form.  

Given a data graph and either a class or an individual node, it creates a shapes graph with one node shape targeting the class or node.  With `cls=`, the shape targets all instances of the class, subclasses included, and the shape and its constraints come from the properties of those class instances.   With `node=`, the shape targets that one node and the shape and its constraints come from the properties of that node.   

The shape is named after its target (`ex:Person` gives `ex:PersonShape`, `ex:alice` gives `ex:AliceShape`) unless `shape=` provides an alternative name. From the values it finds, `derive_shape()` sets the constraints the data supports: `sh:minCount` and `sh:maxCount`, the kind of value (`sh:datatype`, `sh:class` or `sh:nodeKind`) and numeric and string ranges. The shape is `sh:closed`, with `rdf:type` and the descriptive properties are ignored. The derived shape can then used to validate against the same data graph that was used to create it.

**Why use it.** A shape written from scratch can miss how the data is really used. `derive_shape()` gives a starting point drawn from the data itself, which is useful in three situations:

- **There are no shapes yet.** The derived shape shows what the data contains: which properties, how many values, which kinds of value. It can be read, edited and kept as the first version of the shape.
- **Later data should look like earlier data.** A shape derived from data that is known to be good can be used to validate data that arrives later, so that a change in its structure shows up as a violation.
- **The shapes should follow the data.** An existing shape can be given with `template_graph` and `shape=`, and `derive_shape()` brings it up to date with the data, keeping what still holds (section 05.02.3.1).

A derived shape describes the data it was derived from, not what is intended. For example, `sh:maxCount 1` means only that the sample data had one value. Constraints should be reviewed before relying on them as rules.

Below, a shape is derived for the class `ex:Person` and another for the node `ex:alice`, and the data is validated against the class shape (conforms).

In [4]:
import warnings

from starlayer.shacl import derive_shape

observed = StarLayerGraph()
observed.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice" ; ex:age 30 ; ex:knows ex:bob .
    ex:bob a ex:Person ; ex:name "Bob" ; ex:age 41 .
""", format="turtle")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the fidelity warning does not matter here
    # a shape for all the instances of a class
    class_shape = derive_shape(observed, cls=EX.Person)
    # a shape for one node
    node_shape = derive_shape(observed, node=EX.alice)
    # the data conforms to the shape derived from it
    conforms = StarShaclProcessor(shacl_graph=class_shape).validate(data_graph=observed).conforms
print("a shape for the class ex:Person")
show(class_shape)
print("a shape for the node ex:alice")
show(node_shape)
print("the data conforms to its own derived shape:", conforms)


a shape for the class ex:Person
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties (
        <http://purl.org/dc/terms/created>
        <http://purl.org/dc/terms/modified>
        rdf:type
        rdfs:comment
        rdfs:label
    ) ;
    sh:property [
        a sh:PropertyShape ;
        sh:class ex:Person ;
        sh:maxCount 1 ;
        sh:path ex:knows
    ], [
        a sh:PropertyShape ;
        sh:datatype xsd:integer ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:age
    ], [
        a sh:PropertyShape ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:name
    ] ;
    sh:targetClass ex:Person .

a shape for the node ex:alice
@prefi

### 05.02.3.1 Updating a shape: `template_graph`

`derive_shape()` can also start from an existing shape and bring it up to date with the data. It is given `template_graph=` and `shape=`, instead of `cls=` or `node=`.

| What | How it is given |
|---|---|
| the data the shape is derived from | the first argument, as before |
| the shape to update | `shape=`, the IRI of the shape in `template_graph`; the shape's own target says what it is about |
| the earlier version of the shape | `template_graph=`, a shapes graph that holds the template |

`template_graph` is a graph, so `shape=` says which shape in it to update. The shape's own target gives what the shape is about: exactly one `sh:targetClass` (used as `cls=`) or one `sh:targetNode` (used as `node=`). A `ValueError` is raised if `shape=` is missing or not in the template, if the shape does not have exactly one such target, or if `cls=` or `node=` is also passed.

From the data graph, `derive_shape()` reads the relevant parts: the instances of the class (subclasses included) when the target is a class, or the node when it is a node. From `template_graph` it takes the chosen shape and what it references (its property shapes and lists), and the rest of the template is not included in the result.

The result is a new shapes graph with the updated shape. Neither `template_graph` nor the data graph is changed. The shape retains its name from the template.

What happens to each constraint of the template shape depends on the data:

| The template's constraint | The result |
|---|---|
| is consistent with the data | stays as it is, as does everything the data itself cannot decide: `sh:name`, `sh:message`, etc. |
| is broken by the data, and can be worked out again: counts, datatype, class, node kind, ranges, lengths, languages, `sh:in` | is recomputed from the data (e.g. `sh:in` is widened to include the new values) |
| is broken by the data, and cannot be worked out again: `sh:pattern`, `sh:hasValue`, comparisons between two properties such as `sh:equals`, `sh:sparql` and custom constraint components. Also a datatype, class, language list or node kind that the values do not share, and a `sh:minCount` that would come down to zero | is deactivated: it keeps its value and its triple is marked `sh:deactivated true` |
| a property the data has and the template lacks | is added |
| a property the template has and the data lacks | stays, and its `sh:minCount` is deactivated, because no node has a value |

An update loosens a constraint that the data breaks when it can be worked out again, and deactivates it when it cannot. Nothing is dropped, and every change is listed in one `rdfs:comment` on the node shape. It never makes a constraint stricter that the data satisfies: `sh:minCount 2` becomes `1` when the data has one value, and `sh:maxCount 5` stays 5 when the data has two. The result still needs a review before it is relied on.

The template keeps its own setting for `sh:closed`: an open shape stays open, `sh:closed false` stays, and a closed shape stays closed, with its `sh:ignoredProperties` kept. 

Below, an existing `ex:PersonShape` in `template_graph` is brought up to date with the graph `later`, which holds newer data about people. Alice now has two names, so `sh:maxCount 1` on `ex:name` is recomputed as 2, and the `sh:name "Name"` the person wrote stays. Her nickname `Al_1` breaks the `sh:pattern`, which cannot be worked out again, so it is deactivated: the pattern stays, marked `sh:deactivated true`. `ex:age` still holds and is untouched, and `ex:email` is a new property, so it is added. The comment on the shape lists these three changes. The result is shown as Turtle 1.2, which writes the deactivation as an annotation, `{| sh:deactivated true |}`.

In [5]:
template_graph = StarLayerGraph()
template_graph.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:closed true ;
      sh:property [ sh:path ex:name ; sh:datatype xsd:string ; sh:minCount 1 ; sh:maxCount 1 ; sh:name "Name" ] ,
                  [ sh:path ex:age ; sh:datatype xsd:integer ; sh:maxCount 1 ] ,
                  [ sh:path ex:nick ; sh:pattern "^[a-z]+$" ] .
""", format="turtle")

later = StarLayerGraph()
later.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice", "Ally" ; ex:age 30 ; ex:nick "Al_1" ; ex:email "alice@example.org" .
    ex:bob a ex:Person ; ex:name "Bob" ; ex:age 41 .
""", format="turtle")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the fidelity warning does not matter here
    # the data graph, the shape to update, and the graph that holds it
    updated = derive_shape(later, shape=EX.PersonShape, template_graph=template_graph)
print(updated.serialize(format="turtle12", pretty=True, only_used=True))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    rdfs:comment """derive_shape() changed this shape to fit the data:
Added property ex:email: the data has it and the template shape does not.
Loosened ex:name sh:maxCount from 1 to 2: some node has more values.
Deactivated ex:nick sh:pattern "^[a-z]+$": the data breaks it and it cannot be recomputed from the data.""" ;
    sh:closed true ;
    sh:ignoredProperties (
        <http://purl.org/dc/terms/created>
        <http://purl.org/dc/terms/modified>
        rdf:type
        rdfs:comment
        rdfs:label
    ) ;
    sh:property [
        a sh:PropertyShape ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
        sh:path ex:email
    ], [
        sh:datatype xsd:integer ;
      

## 05.02.4 Checking a shapes graph: `meta_validate()`

`meta_validate()` checks a shapes graph against SHACL's own shapes to test whether it is well formed. It is a function and needs no processor, so it can check shapes before they are saved or used.

`meta_validate()` is an ordinary SHACL validation. The shapes graph is the data graph, and the shapes are `shacl_shacl`, the SHACL 1.2 shapes for shapes (`starontology.get_ontology_graph("shacl_shacl")`, from `starlayer/starontology/shacl/shacl12-shacl.ttl`). The result is a standard validation report containing `conforms`, the `report_graph` and the `report_text`. Reading a validation report, its results and its severities, is covered in [05.01.4](05.01-validating-with-shacl-1.2.ipynb).

- A valid shapes graph returns a report with `conforms` true.
- A shapes graph that is not valid raises a `MetaShaclError`, which carries the report, with `conforms` false.The error also has a `problems` list, one line for each result in the report that describes the issue in english, and a `scope`, which says what the check covered (section 05.02.4.2).

A processor created with `meta_shacl=True` performs the same validation when the processor is created, and raises the same `MetaShaclError`. It is often not necessary to validate shapes when using a processor because shapes that are checked prior to publication do not have to be checked each time they are used. It may be useful to use `meta_shacl=True` when the processor is being used while the shapes themselves are being edited.

Below, a valid shapes graph passes, and a shapes graph with two `sh:datatype` values on one property shape raises a `MetaShaclError`. The error holds the problem and the report.

In [10]:
from starlayer.shacl import MetaShaclError, meta_validate

valid = StarLayerGraph()
valid.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")
report = meta_validate(valid)
print("valid shapes:", report.conforms)
print()

two_datatypes = StarLayerGraph()
two_datatypes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:datatype xsd:string, xsd:integer ] .
""", format="turtle")
try:
    meta_validate(two_datatypes)
except MetaShaclError as error:
    print(error.report_text)
    print("problems:", error.problems)

valid shapes: True

Validation Report
Conforms: False
Results (1):
Violation in MaxCountConstraintComponent (e37ec35afb3773e4650ff1873337bb4c):
	Focus Node: e054322ea55dd6bf3f840582a64b9057
	Message: Allows at most 1 value, found 2
problems: ('the property shape of ex:PersonShape for ex:name, sh:datatype: Allows at most 1 value, found 2',)


### 05.02.4.1 Adding custom checks: `meta_shapes_extra`

`meta_validate()` takes `meta_shapes_extra`, additional shapes to check the shapes graph against, along with SHACL's own. They are ordinary shapes, applied with the shapes graph as their data graph. A rule about how shapes are written is a shape that targets `sh:NodeShape` or `sh:PropertyShape`. `meta_shapes_extra` is a way to apply a custom rule to shapes graphs, such as requiring every node shape to have a label. A processor created with `meta_shacl=True` takes the same setting. 

Below, a shape requires every node shape to have an `rdfs:label`. A shapes graph without labels passes SHACL's own check but fails this one, and one with a label passes.

In [ ]:
extra = StarLayerGraph()
extra.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:LabelRule a sh:NodeShape ;
      sh:targetClass sh:NodeShape ;
      sh:property [ sh:path rdfs:label ; sh:minCount 1 ; sh:message "A node shape needs an rdfs:label" ] .
""", format="turtle")

unlabelled = StarLayerGraph()
unlabelled.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person .
""", format="turtle")

labelled = StarLayerGraph()
labelled.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:PersonShape a sh:NodeShape ; rdfs:label "Person" ; sh:targetClass ex:Person .
""", format="turtle")

meta_validate(unlabelled)  # SHACL's own shapes accept it
print("SHACL's own shapes accept the shapes graph without a label")
try:
    meta_validate(unlabelled, meta_shapes_extra=[extra])
except MetaShaclError as error:
    print("with the extra shape:", error.problems)
meta_validate(labelled, meta_shapes_extra=[extra])
print("with a label: accepted")

SHACL's own shapes accept the shapes graph without a label
with the extra shape: ('ex:PersonShape, rdfs:label: A node shape needs an rdfs:label',)
with a label: accepted


### 05.02.4.2 Checking an edit before it is made

An editor often needs to know whether an edit would leave the shapes graph valid before the edit is made. `meta_validate(shapes_graph, add=..., remove=...)` answers that question. `add` and `remove` are sets of triples of the proposed edit. The shapes graph is validated as it would be after the edits. The shapes graph itself is not changed.

To keep the check fast, only part of the graph is checked: the nodes the edit changes, and the shapes that refer to them. On a large shapes graph this takes a fraction of a second, where checking the whole graph could take considerably longer. An edit that can change how other nodes are read, such as a prefix declaration or a constraint component, is checked in full automatically. The option `full=True` validates the whole graph regardless of the nature of the change.

`meta_scope()` says what a check would cover, without running it. It returns a `MetaScope`: `full` is true when the whole graph would be checked, `nodes` lists the nodes it would check, and `reasons` says why. A `MetaShaclError` carries the scope that was checked in `scope`.

A check of part of the graph is useful when editing a shapes graph. The full `meta_validate()` should be run before the shapes are placed into production.

Below, an edit that adds a second `sh:datatype` is refused before it is made, and the graph is unchanged. `meta_scope()` shows that this edit is checked on its own nodes only, while an edit to a prefix declaration needs the whole graph. Replacing the datatype, with the old triple removed and the new one added, is accepted. Last, a harmless edit is checked in full with `full=True`.

In [ ]:
from starlayer.graph import Literal, Namespace, StarLayerGraph
from starlayer.shacl import SH, MetaShaclError, meta_scope, meta_validate

EX = Namespace("http://example.org/")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")
current = StarLayerGraph()
current.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:property ex:AgeShape .
    ex:AgeShape sh:path ex:age ; sh:datatype xsd:integer .
""", format="turtle")

second_datatype = [(EX.AgeShape, SH.datatype, XSD.string)]  # an edit not yet made
try:
    meta_validate(current, add=second_datatype)
except MetaShaclError as error:
    print("edit refused:", error.scope.reasons)
print("the graph is unchanged:", len(list(current.triples((EX.AgeShape, SH.datatype, None)))), "datatype")

# replacing a value: remove the old triple and add the new one in the same call
integer_datatype = (EX.AgeShape, SH.datatype, XSD.integer)
report = meta_validate(current, remove=[integer_datatype], add=second_datatype)
print("replacing the datatype is valid:", report.conforms)

prefix_edit = [(EX.PersonShape, SH.declare, EX.Prefixes)]
print("full check for the datatype edit:", meta_scope(current, add=second_datatype).full)
print("a prefix edit:", meta_scope(current, add=prefix_edit).reasons)

name_edit = [(EX.AgeShape, SH.name, Literal("Age"))]  # a harmless edit
print("full=True forces the whole graph:", meta_scope(current, add=name_edit, full=True).full)
report = meta_validate(current, add=name_edit, full=True)
print("the harmless edit, checked in full:", report.conforms)

edit refused: ('checked 3 nodes: 2 changed, 1 dependent',)
the graph is unchanged: 1 datatype
replacing the datatype is valid: True
full check for the datatype edit: False
a prefix edit: ('full check: sh:declare changed on ex:PersonShape',)
full=True forces the whole graph: True
the harmless edit, checked in full: True


## 05.02.5 Errors

A SHACL engine can raise various kinds of errors: errors in the shapes graph, execution errors, and errors in the function or method call. A call is the function the application invokes, and its arguments are checked first. A run is what the engine does once it has accepted them: reading the shapes graph, validating, evaluating and applying rules. Each message says which shape, property, rule or argument gave rise to the error.

`ShapeError` and `ExecutionError` both derive from `StarLayerError` (in `starlayer.errors`), so `except StarLayerError` catches any error of the engine. Neither is a `ValueError`. `TypeError` and `ValueError` are Python's own, raised for a bad call.

| What is wrong | Error | Raised when | It carries | Discussed in |
|---|---|---|---|---|
| The shapes graph | `ShapeError` | The base of the two errors below, for any problem with the shapes graph. | Its subclass carries more. | This section |
| The shapes graph | `MetaShaclError` | The shapes graph does not conform to SHACL's own shapes when validated with `meta_validate()`. | A standard SHACL validation report, along with a set of human-readable `problems` and `scope` of the validation. | section 05.02.4 |
| The shapes graph | `ShapeCompileError` | The engine cannot read or compile the shapes graph, its rules or its setup, for example a `sh:minCount "many"`, an invalid regular expression or a rule that cannot be read. Most of these can be detected before compiling with `meta_validate()` (section 05.02.4). Other aspects of a shapes graph, such as an `owl:imports` that cannot be resolved, can only be evaluated when compiling. These errors typically arise when the processor is created, or otherwise by the first `validate()`, `evaluate()` or `apply_rules()` that finds the mistake. | A message that names the shape and the property. | [05.01.4.2](05.01-validating-with-shacl-1.2.ipynb) |
| A run | `ExecutionError` | The shapes graph compiles correctly but running it failed. The `kind` says how. | `kind`, a message that names the shape, rule or expression, and `context`, which says where it happened. | The table below |
| A call | `TypeError` | A call is given a value of the wrong type. It is Python's own error and a mistake in the calling code, not a SHACL error. For example, a graph of the wrong type (an rdflib graph) or an option the function does not take, such as `allow_warnings` on `apply_rules()`. It is raised before the engine starts. | A message that names the argument. | The example below |
| A call | `ValueError` | An argument is of the right type but has a value the function refuses. For example, the `role` argument of `select_widgets()` that is not `'editor'` or `'viewer'`. It is raised before the engine starts. A value error that arises while the shapes are compiled or run is a `ShapeCompileError` or an `ExecutionError` instead. | A message that names the argument. | The guide of the function |

The `kind` of an `ExecutionError`:

| `kind` | Raised when | Discussed in |
|---|---|---|
| `"rule"` | A rule fails while running, or `max_iterations` or `max_triples` is exceeded. The rule set fails as a whole. | [05.06](05.06-shacl-inference-rules.ipynb) |
| `"expression"` | A node expression cannot be evaluated, which is an error and not an empty result: for example a SPARQL function given arguments it cannot take, such as a division by zero. `evaluate_expression()` raises it. | [05.10](05.10-building-on-shacl.ipynb) |
| `"nesting"` | Shapes or node expressions nest too deep: a validation path with more nested shapes than the limit, or a node expression that calls itself, directly or through a function. | This section |
| `"recursion"` | A shape reaches itself on the same focus node through `sh:not`, which happens when the data has a cycle. | This section |
| `"constraint-query"` | A `sh:sparql` constraint query reports `?failure true`. | This section |
| `"internal"` | The engine failed an assertion about its own result. This likely indicates a bug in StarLayer itself. | This section |
| `"timeout"` | A reasoner ran past its time limit (the `timeout` of an entailment regime such as OWL-Direct). | [02.07](02.07-owl-dl-reasoning.ipynb) |
| `"inconsistent"` | The ontology is inconsistent, so the reasoner cannot answer (OWL-Direct). | [02.07](02.07-owl-dl-reasoning.ipynb) |
| `"inference"` | A reasoner could not run: Java is missing or broken, or the reasoner process ended without a result. | [02.07](02.07-owl-dl-reasoning.ipynb) |

The `kind` names the cause and does not change with where the failure happens: an expression that fails inside a rule is still kind `"expression"`, and the rule is named in the message and in the error's optional `context`.

**`ShapeCompileError` and `MetaShaclError`.** Both are about a shapes graph that is not valid, and they overlap. A `ShapeCompileError` comes from the engine itself, which stops at the first error it encounters. A `MetaShaclError` comes from validating the shapes graph against SHACL's own shapes: it reports every problem at once. It also finds mistakes in shapes that the engine can read and run, but may generate inconsistent results, such as two `sh:datatype` values on one shape. Running `meta_validate()` on a shapes graph before validating finds most mistakes that would otherwise be a `ShapeCompileError`. 

Some mistakes cannot be found this way because they depend on the data graph or on the run. For example, a shape that reaches itself through `sh:not` is a mistake only when the data has a cycle, and the same shapes are acceptable on a chain. The shapes graph is not wrong, but the run fails, so this raises an `ExecutionError` of kind `"recursion"`.

Catch `ExecutionError` for the failure of a run, and use its `kind` to identify the cause. Neither `ExecutionError` nor `ShapeError` is a `ValueError`, so `except ValueError` catches neither.

Below, a shape with `sh:minCount "many"` is refused by `meta_validate()` with a `MetaShaclError` that describes the problem. The same shapes graph raises a `ShapeCompileError` when the processor compiles it without fixing the error. A shape with two `sh:datatype` values is refused by `meta_validate()`, but the engine compiles it without an error. A shape that reaches itself through `sh:not` passes `meta_validate()`, conforms on a chain, and raises an `ExecutionError` of kind `"recursion"` on a cycle. A division by zero in a node expression raises one of kind `"expression"`, and an option that `apply_rules()` does not take raises a `TypeError`.

In [9]:
from starlayer.graph import Namespace, StarLayerGraph
from starlayer.shacl import ExecutionError, MetaShaclError, ShapeCompileError, ShapeError, StarShaclProcessor, apply_rules, evaluate_expression, meta_validate

EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")

def shapes(turtle):
    graph = StarLayerGraph()
    graph.parse(data="@prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n" + turtle, format="turtle")
    return graph

def first_line(error):
    return str(error).splitlines()[0]

# a value of the wrong kind: the function meta_validate() and compiling find the same mistake
unreadable = shapes('ex:S a sh:NodeShape ; sh:property [ sh:path ex:p ; sh:minCount "many" ] .')
print('A value of the wrong kind: sh:minCount "many"')
try:
    meta_validate(unreadable)
except MetaShaclError as error:
    print("  meta_validate():", "MetaShaclError:", error.problems[0])
try:
    StarShaclProcessor(shacl_graph=unreadable)
except ShapeCompileError as error:
    print("  compiling:      ", "ShapeCompileError:", first_line(error))

# a shape the engine can read but SHACL's own shapes reject: only the meta check finds it
two_datatypes = shapes('ex:S a sh:NodeShape ; sh:property [ sh:path ex:p ; sh:datatype xsd:string, xsd:integer ] .')
print()
print("Two sh:datatype values on one shape")
try:
    meta_validate(two_datatypes)
except MetaShaclError as error:
    print("  meta_validate():", "MetaShaclError:", error.problems[0])
StarShaclProcessor(shacl_graph=two_datatypes)
print("  compiling:       no error, the engine accepts it")

# both are ShapeErrors: one except clause covers them
print()
print("One except clause for both")
for check in (lambda: StarShaclProcessor(shacl_graph=unreadable), lambda: meta_validate(two_datatypes)):
    try:
        check()
    except ShapeError as error:
        print("  caught as ShapeError:", type(error).__name__)

# a mistake only the data shows: a shape that reaches itself through sh:not
recursive = shapes('ex:N a sh:NodeShape ; sh:targetNode ex:a ; sh:not [ sh:property [ sh:path ex:next ; sh:node ex:N ] ] .')
meta_validate(recursive)   # the shapes graph is valid: there is nothing for the shapes to say
processor = StarShaclProcessor(shacl_graph=recursive)
print()
print("A shape that reaches itself through sh:not")
print("  meta_validate():  no error")
print("  on a chain:       conforms =", processor.validate(shapes("ex:a ex:next ex:b .")).conforms)
try:
    processor.validate(shapes("ex:a ex:next ex:a ."))
except ExecutionError as error:
    print("  on a cycle:       ExecutionError, kind", repr(error.kind) + ":", first_line(error))

# a node expression that cannot be evaluated: a SPARQL function given arguments it cannot take
holder = shapes("ex:holder ex:expression [ <http://www.w3.org/ns/sparql#divide> ( 1 0 ) ] .")
expression = next(holder.objects(EX.holder, EX.expression))
print()
print("A node expression that divides by zero")
try:
    evaluate_expression(holder, StarLayerGraph(), expression, EX.focus)
except ExecutionError as error:
    print("  ExecutionError, kind", repr(error.kind) + ":", error)

# not a SHACL error: an option the function does not take
print()
print("An option the function does not take")
try:
    apply_rules(StarLayerGraph(), shapes("ex:S a sh:NodeShape ."), allow_warnings=False)
except TypeError as error:
    print("  TypeError:", first_line(error).split(":")[0])

A value of the wrong kind: sh:minCount "many"
  meta_validate(): MetaShaclError: the property shape of ex:S for ex:p, sh:minCount: Value "many" should have the datatype xsd:integer
  compiling:       ShapeCompileError: the property shape of http://example.org/S for http://example.org/p: sh:minCount needs exactly one non-negative xsd:integer, got ['many']

Two sh:datatype values on one shape
  meta_validate(): MetaShaclError: the property shape of ex:S for ex:p, sh:datatype: Allows at most 1 value, found 2
  compiling:       no error, the engine accepts it

One except clause for both
  caught as ShapeError: ShapeCompileError
  caught as ShapeError: MetaShaclError

A shape that reaches itself through sh:not
  meta_validate():  no error
  on a chain:       conforms = True
  on a cycle:       ExecutionError, kind 'recursion': recursive shape the sh:not of http://example.org/N reaches itself on http://example.org/a through sh:not

A node expression that divides by zero
  ExecutionError, kin

## 05.02.6 Descriptive shape properties

Shapes can contain properties that describe a shape and have no impact on validation or other operations. People and tools can read the descfriptive properties, but a validating processor should ignore them. The descriptive properties of SHACL are outlined below.  

- `sh:name` and `sh:description` name and describe what the shape is about: the property of a property shape, or the kind of resource a node shape targets. A form may use them as the field label and help text.
- `rdfs:label` and `rdfs:comment` name and describe the shape resource itself. Note that for a `sh:ShapeClass`, these are `rdfs:label` and `rdfs:comment` properties of the class as well.
- `sh:intent` states in words a rule the shape is meant to express, including rules the shape cannot check.
- `sh:unit` gives the unit of measure or currency of the values, as a UCUM or ISO 4217 code or an IRI from a units vocabulary. It describes the values but is not validated. 
- `sh:agentInstruction` is a text instruction for a software agent that reads the shape.
- `sh:codeIdentifier` is the name the shape gets in generated code, an API, or a table or column name. It is one string of letters, digits and underscores, not starting with a digit. `sh:codeType` is the datatype for the code to use for values. Both are merely hints. SHACL does not convert values.
- `sh:order` is the general order of a shape among its siblings. It is often used to sort property shapes and property groups in a form.


Below, a node shape and a property shape use these properties. 


```turtle
@prefix ex:   <http://example.org/> .
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    sh:name "Person" ;                                  # the kind of resource the shape describes
    sh:description "A living individual that the organisation keeps records about." ;
    rdfs:label "Person shape" ;                         # the shape resource itself
    rdfs:comment "The shape that checks the data held about a person." ;
    sh:intent "A person's data is kept for as long as they are a customer." ;
    sh:codeIdentifier "customer" ;                      # the table, in a generated SQL schema
    sh:targetClass ex:Person ;
    sh:property ex:Person-weight .

ex:Person-weight a sh:PropertyShape ;
    sh:name "Weight" ;                                  # the property path, ex:weight
    sh:description "The person's body weight." ;
    sh:intent "Each person has a weight." {| sh:formalized true |} ;
    sh:intent "The weight is at least 20 kg." {| sh:formalized true |} ;
    sh:intent "The weight is measured, not estimated." ;   # a rule the shape cannot check
    sh:order 2 ;
    sh:unit "kg" ;
    sh:codeIdentifier "weight_kg" ;                     # the column
    sh:codeType xsd:decimal ;
    sh:agentInstruction "Do not estimate a weight that is missing. Ask for it." ;
    sh:path ex:weight ;
    sh:minCount 1 ;
    sh:datatype xsd:decimal ;
    sh:minInclusive 20 .
```